### Read experimental result data

In [32]:
# read parquet file
import polars as pl
# df = pl.read_parquet("../test/result.parquet")
df = pl.read_parquet("../../experiments/16_10__result.parquet")
print(df)

# read metadata of the parquet file
import pyarrow.parquet as pq
# meta = pq.read_metadata("../test/result.parquet")
meta = pq.read_metadata("../../experiments/16_10__result.parquet")

# You can check versions depending on libraries 
print(meta.metadata)

shape: (500_000, 5)
┌─────┬─────┬───────┬───────┬───────────────┐
│ id  ┆ ip  ┆ us_id ┆ msg   ┆ mean_num_xact │
│ --- ┆ --- ┆ ---   ┆ ---   ┆ ---           │
│ u64 ┆ u64 ┆ u64   ┆ str   ┆ f64           │
╞═════╪═════╪═══════╪═══════╪═══════════════╡
│ 0   ┆ 329 ┆ 0     ┆ 01000 ┆ 13.48         │
│ 0   ┆ 329 ┆ 0     ┆ 11000 ┆ 27.73         │
│ 0   ┆ 329 ┆ 0     ┆ 30000 ┆ 49.9          │
│ 0   ┆ 329 ┆ 0     ┆ 20000 ┆ 37.46         │
│ 0   ┆ 329 ┆ 0     ┆ 40000 ┆ 50.77         │
│ …   ┆ …   ┆ …     ┆ …     ┆ …             │
│ 159 ┆ 246 ┆ 9     ┆ 43444 ┆ 108.81        │
│ 159 ┆ 246 ┆ 9     ┆ 03444 ┆ 60.54         │
│ 159 ┆ 246 ┆ 9     ┆ 34444 ┆ 93.06         │
│ 159 ┆ 246 ┆ 9     ┆ 04444 ┆ 61.95         │
│ 159 ┆ 246 ┆ 9     ┆ 44444 ┆ 104.43        │
└─────┴─────┴───────┴───────┴───────────────┘
{b'version_runner': b'0.1.1', b'version_core': b'0.1.1', b'version': b'0.1.0', b'ARROW:schema': b'/////0wBAAAQAAAAAAAKAAwACgAJAAQACgAAABAAAAAAAQQACAAIAAAABAAIAAAABAAAAAUAAADsAAAAsAAAAHAAAABEAAAABAAA

### Compute solutions and/or values of optimal, RAM, and URS policies

In [34]:
EV_BY_OPT = "ev_by_opt"
EV_BY_RAM = "ev_by_ram"
EV_BY_URS = "ev_by_urs"
OPT_MEAN = "mean_of_opt_values"
RAMEV_OPTEV_RATIO = "ram ev / opt ev"
URSEV_OPTEV_RATIO = "urs ev / opt ev"
RAMEV_OPTM_RATIO = "ram ev / opt mean"
URSEV_OPTM_RATIO = "urs ev / opt mean"

MEAN_NUM_XACT = "mean_num_xact"

lf_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", MEAN_NUM_XACT])
    .select([pl.col("us_id"), pl.col("ip"), pl.col("msg").name.prefix("argmax_"), pl.col(MEAN_NUM_XACT).name.prefix("max_")])
)
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))
    .group_by("ip_right", "us_id_right")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])
    .select([MEAN_NUM_XACT, "max_" + MEAN_NUM_XACT])
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias(EV_BY_OPT), pl.col("max_" + MEAN_NUM_XACT).alias(OPT_MEAN)])
)
lf_rob = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max())
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_RAM))
)
lf_urs = (
    df.lazy()
    .select(MEAN_NUM_XACT)
    .mean()
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_URS))
)
eval_df = (
    pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")
    .with_columns([
        (pl.col(EV_BY_RAM) / pl.col(EV_BY_OPT)).alias(RAMEV_OPTEV_RATIO),
        (pl.col(EV_BY_URS) / pl.col(EV_BY_OPT)).alias(URSEV_OPTEV_RATIO),
        (pl.col(EV_BY_RAM) / pl.col(OPT_MEAN)).alias(RAMEV_OPTM_RATIO),
        (pl.col(EV_BY_URS) / pl.col(OPT_MEAN)).alias(URSEV_OPTM_RATIO),
    ])
).collect().transpose(include_header=True).rename({"column": "key", "column_0": "value"})

In [35]:
eval_df

key,value
str,f64
"""ev_by_opt""",160.527381
"""mean_of_opt_values""",175.589375
"""ev_by_ram""",170.516875
"""ev_by_urs""",81.504842
"""ram ev / opt ev""",1.062229
"""urs ev / opt ev""",0.507732
"""ram ev / opt mean""",0.971112
"""urs ev / opt mean""",0.464179


In [41]:
#optimal strategy for each model of diffusion
lf_opt.sort(pl.col("max_mean_num_xact"),descending = True).collect()

us_id,ip,argmax_msg,max_mean_num_xact
u64,u64,str,f64
8,130,"""44100""",376.36
8,246,"""44200""",365.85
8,113,"""44100""",364.65
1,113,"""44100""",346.54
7,113,"""44000""",346.45
…,…,…,…
5,329,"""42001""",21.33
0,62,"""41001""",17.61
3,62,"""41004""",14.88


In [117]:
###ram
lf_rob_details = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .sort(by="mean_num_xact", descending=True)
    # 上位5行を取得
).collect()
lf_rob_details[:10]

msg,mean_num_xact
str,f64
"""44000""",170.516875
"""43000""",166.319688
"""42000""",162.03375
"""44100""",161.453937
"""44001""",160.882688
"""43100""",159.102687
"""41000""",158.029562
"""43001""",157.881813
"""42100""",155.380063


In [38]:
# read parquet file
import polars as pl
user_analysis= pl.read_parquet("../../experiments/user_analysis.parquet")
user_analysis


ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
329,0,"""01000""",0,0.0,0.0
329,0,"""01000""",1,0.0,0.04
329,0,"""01000""",2,0.0,0.0
329,0,"""01000""",3,0.0,0.0
329,0,"""01000""",4,0.0,0.14
…,…,…,…,…,…
246,9,"""44444""",470,0.98,0.0
246,9,"""44444""",471,0.0,0.0
246,9,"""44444""",472,0.75,0.53


In [32]:
user_analysis

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
315,0,"""120""",0,0.85,0.86
315,0,"""120""",1,0.84,1.44
315,0,"""120""",2,0.99,0.75
315,0,"""120""",3,1.0,1.0
315,0,"""120""",4,1.0,2.52
…,…,…,…,…,…
163,1,"""102""",470,0.0,0.0
163,1,"""102""",471,0.0,0.72
163,1,"""102""",472,0.0,0.0


In [93]:
test1 = (user_analysis.lazy().filter((pl.col("ip") == 130)&(pl.col("us_id") == 8)&(pl.col("msg") == "44100"))
        .select([pl.col("num_xact_of_users").sum(),pl.col("num_share_of_users").sum()]).collect())
test2 = (user_analysis.lazy().filter((pl.col("ip") == 130)&(pl.col("us_id") == 8)&(pl.col("msg") == "44100"))
        .select([pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean()]).collect())

In [94]:
test2

num_xact_of_users,num_share_of_users
f64,f64
0.792337,1.095895


元のデータ


ip=1, us_id=0, msg="200"の場合の外部行動者数

In [48]:
(df.lazy().filter(pl.col("ip") == 163).filter(pl.col("us_id") == 0).filter(pl.col("msg") == "200")).collect()

id,ip,us_id,msg,mean_num_xact
u64,u64,u64,str,f64
1,163,0,"""200""",278.9


In [50]:
df.lazy().filter(pl.col("msg") == "200").collect()

id,ip,us_id,msg,mean_num_xact
u64,u64,u64,str,f64
0,315,0,"""200""",441.37
1,163,0,"""200""",278.9
2,315,1,"""200""",453.65
3,163,1,"""200""",221.47


一人ずつ抽出したユーザーの外部行動回数

ロバストな戦略であるメッセージシーケンス :

"44000"	170.516875

"43000"	166.319688

"42000"	162.03375

"44100"	161.453937

"44001"	160.882688

In [129]:
ram_msg = (lf_rob_details.lazy().select(pl.col("msg")).collect())
ram_msg_list = ram_msg["msg"][:10].to_list()
ram_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).collect()

reverse_msg_list = ram_msg["msg"][-10:].to_list()
reverse_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).collect()
reverse_msg_us_an

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
329,0,"""00433""",0,0.0,0.0
329,0,"""00433""",1,0.0,0.0
329,0,"""00433""",2,0.0,0.0
329,0,"""00433""",3,0.0,0.0
329,0,"""00433""",4,0.0,0.03
…,…,…,…,…,…
246,9,"""00444""",470,0.95,0.14
246,9,"""00444""",471,0.0,0.0
246,9,"""00444""",472,0.54,0.99


In [135]:
ram_msg_us_an.lazy().filter(pl.col("num_share_of_users") > 4).collect()

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
177,0,"""41000""",367,0.84,4.18
177,0,"""41000""",450,0.88,4.39
177,0,"""42000""",367,0.86,4.07
177,0,"""42000""",450,0.83,4.27
177,0,"""43000""",367,0.88,4.08
…,…,…,…,…,…
113,9,"""41000""",266,0.99,4.3
113,9,"""42000""",266,1.0,4.19
113,9,"""43000""",266,0.97,4.08


In [131]:
reverse_msg_us_an.write_parquet("../../experiments/reverse_msg_us_an.parquet")

In [127]:
ram_msg_us_an.write_parquet("../../experiments/ram_msg_us_an.parquet")

In [124]:
ram_ratio = (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
ram_mean_xact_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
ram_mean_share_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
ram = (
    pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())
ram


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.338118,0.391654,0.863309


逆の順番に情報を提供すると

In [76]:
# re_ram_xact_mean= user_analysis.lazy().filter(pl.col("msg") == "00044").select(pl.col("num_xact_of_users").mean())
# re_ram_share_mean= user_analysis.lazy().filter(pl.col("msg") == "00044").select(pl.col("num_share_of_users").mean())
# re_ram_xact_mean/re_ram_share_mean.alias()
ratio_00044 = (user_analysis.lazy().filter(pl.col("msg") == "00044")
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
mean_xact_of_users_00044=user_analysis.lazy().filter(pl.col("msg") == "00044").select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
mean_share_of_users_00044=user_analysis.lazy().filter(pl.col("msg") == "00044").select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
_00044 = (
    pl.concat([mean_xact_of_users_00044, mean_share_of_users_00044, ratio_00044], how="horizontal").collect())
_00044

mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.06643,0.329714,0.201478


最適戦略

拡散モデルip,us_idの組み合わせに、それに向ける最適解のmsgを同時に適用した全てのリストのユーザー状態

In [102]:
opt_df = (
    df.lazy()
    .filter(pl.col("mean_num_xact") == pl.col("mean_num_xact").max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", "mean_num_xact"])
    .sort(pl.col("mean_num_xact"),descending = True)
    .select(pl.col("us_id"),pl.col("ip"),pl.col("msg"))
)
# 2. 結合の「キー」として使う LazyFrame を準備
#    opt_df の "argmax_msg" を user_analysis の "msg" に名前を合わせる
opt_keys = opt_df.lazy().select(
    pl.col("us_id"),
    pl.col("ip"),
    pl.col("msg")
)
# 3. user_analysis を 'semi' join でフィルタリングする
opt_ratio_lazy = user_analysis.lazy().join(
    opt_keys,
    on=["us_id", "ip", "msg"],  # 3つのキーがすべて一致する行を探す
    how="semi"                  # user_analysis 側に存在する行だけを残す
).collect()
opt_ratio_lazy


ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
329,0,"""41000""",0,0.0,0.0
329,0,"""41000""",1,0.0,0.03
329,0,"""41000""",2,0.02,0.0
329,0,"""41000""",3,0.0,0.0
329,0,"""41000""",4,0.0,0.06
…,…,…,…,…,…
246,9,"""44000""",470,1.0,1.19
246,9,"""44000""",471,0.56,0.56
246,9,"""44000""",472,0.99,1.48


各拡散モデルにおけるOptimalのmsgの結果のユーザー状態

ユーザー1人の平均外部・内部行動回数と外部対内部の比率

In [103]:
opt_ratio = (opt_ratio_lazy.lazy().group_by(["msg","us_id","ip"])
        .agg(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"),
             pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"),
            (pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")
        )
        .collect())
opt_ratio

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,u64,u64,f64,f64,f64
"""44012""",2,266,0.291242,0.248484,1.172075
"""43000""",7,130,0.653221,0.865684,0.754572
"""42000""",4,42,0.088526,0.090968,0.973154
"""43100""",7,286,0.593958,0.701284,0.846957
"""44000""",2,205,0.512042,0.614189,0.833688
…,…,…,…,…,…
"""44000""",1,42,0.226063,0.204126,1.107467
"""43000""",5,291,0.247558,0.221684,1.116714
"""44000""",7,153,0.501979,0.544379,0.922113


In [136]:
opt_ratio.lazy().select(pl.col("mean_num_xact_of_users").mean()).collect()

mean_num_xact_of_users
f64
0.369662


最適解OPTとして扱われたmsgの頻度

In [154]:
opt_msg_counts =(
opt_ratio.lazy().group_by("msg").agg(pl.len().alias("msg_count")).sort("msg_count",descending=True).collect()
)
opt_msg_list = opt_msg_counts.select(pl.col("msg").head(10))
opt_msg_counts

msg,msg_count
str,u32
"""44000""",56
"""43000""",16
"""44100""",14
"""44001""",10
"""42000""",9
…,…
"""44200""",1
"""44014""",1
"""42002""",1


In [138]:
ttt =opt_ratio.lazy().filter(pl.col("msg") == "44000")
# ,(pl.col("mean_num_share_of_users").mean()),(pl.col("mean_xact_of_users").mean() / pl.col("mean_share_of_users").mean()).alias("xact_share_ratio"))
# .collect()

ttt.mean().collect()

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,f64,f64,f64,f64,f64
null,4.017857,224.678571,0.455427,0.492116,0.9829


In [152]:
opt = (opt_ratio_lazy.lazy().filter(pl.col("msg").is_in(opt_msg_list["msg"].to_list()))).collect()
opt.write_parquet("../../experiments/opt.parquet")

ロバストな戦略と最適戦略におけるユーザーの外部・内部行動回数の傾向

ロバストな戦略に該当する200の場合、ユーザー1人の外部行動回数の平均: 0.734416, ユーザー1人の内部行動回数の平均: 1.194905
        



最適戦略に該当する210の場合

ユーザー1人の外部行動回数の平均: 0.671784, 内部行動回数の平均:0.994447



両方の戦略とも内部行動の回数が常に外部行動の回数より上回っている

In [96]:
##opt
lf_opt.sort("max_mean_num_xact",descending=True).collect()

us_id,ip,argmax_msg,max_mean_num_xact
u64,u64,str,f64
8,130,"""44100""",376.36
8,246,"""44200""",365.85
8,113,"""44100""",364.65
1,113,"""44100""",346.54
7,113,"""44000""",346.45
…,…,…,…
5,329,"""42001""",21.33
0,62,"""41001""",17.61
3,62,"""41004""",14.88


予測した拡散モデルと対応する最適戦略を同時に適用した場合

In [4]:
us_210_1=user_analysis.lazy().filter((pl.col("msg") == "210") & (pl.col("us_id") == 1) & (pl.col("ip") == 315))
us_210_2=user_analysis.lazy().filter((pl.col("msg") == "210") & (pl.col("us_id") == 0) & (pl.col("ip") == 315))
us_200_1 =user_analysis.lazy().filter((pl.col("msg") == "200") & (pl.col("us_id") == 0) & (pl.col("ip") == 163))
us_200_2 = user_analysis.lazy().filter((pl.col("msg") == "201") & (pl.col("us_id") == 1) & (pl.col("ip") == 163))
# 各ユーザーの num_xact_of_users 分布を確認
print(us_210_1.select("num_xact_of_users").describe(),
us_210_1.select("num_share_of_users").describe())

shape: (9, 2)
┌────────────┬───────────────────┐
│ statistic  ┆ num_xact_of_users │
│ ---        ┆ ---               │
│ str        ┆ f64               │
╞════════════╪═══════════════════╡
│ count      ┆ 475.0             │
│ null_count ┆ 0.0               │
│ mean       ┆ 0.992421          │
│ std        ┆ 0.05813           │
│ min        ┆ 0.0               │
│ 25%        ┆ 1.0               │
│ 50%        ┆ 1.0               │
│ 75%        ┆ 1.0               │
│ max        ┆ 1.0               │
└────────────┴───────────────────┘ shape: (9, 2)
┌────────────┬────────────────────┐
│ statistic  ┆ num_share_of_users │
│ ---        ┆ ---                │
│ str        ┆ f64                │
╞════════════╪════════════════════╡
│ count      ┆ 475.0              │
│ null_count ┆ 0.0                │
│ mean       ┆ 1.724632           │
│ std        ┆ 0.717614           │
│ min        ┆ 0.0                │
│ 25%        ┆ 1.12               │
│ 50%        ┆ 1.88               │
│ 75%        ┆ 

In [144]:
print(us_200_2.select("num_xact_of_users").describe()
,us_200_2.select("num_share_of_users").describe())

shape: (9, 2)
┌────────────┬───────────────────┐
│ statistic  ┆ num_xact_of_users │
│ ---        ┆ ---               │
│ str        ┆ f64               │
╞════════════╪═══════════════════╡
│ count      ┆ 475.0             │
│ null_count ┆ 0.0               │
│ mean       ┆ 0.507853          │
│ std        ┆ 0.332339          │
│ min        ┆ 0.0               │
│ 25%        ┆ 0.0               │
│ 50%        ┆ 0.7               │
│ 75%        ┆ 0.78              │
│ max        ┆ 0.91              │
└────────────┴───────────────────┘ shape: (9, 2)
┌────────────┬────────────────────┐
│ statistic  ┆ num_share_of_users │
│ ---        ┆ ---                │
│ str        ┆ f64                │
╞════════════╪════════════════════╡
│ count      ┆ 475.0              │
│ null_count ┆ 0.0                │
│ mean       ┆ 0.414526           │
│ std        ┆ 0.579473           │
│ min        ┆ 0.0                │
│ 25%        ┆ 0.0                │
│ 50%        ┆ 0.0                │
│ 75%        ┆ 

In [5]:

us_200_1.select("num_xact_of_users").describe()


statistic,num_xact_of_users
str,f64
"""count""",475.0
"""null_count""",0.0
"""mean""",0.587158
"""std""",0.262615
"""min""",0.0
"""25%""",0.27
"""50%""",0.72
"""75%""",0.81
"""max""",0.93


In [30]:
import pingouin as pg

msg_200 = (user_analysis.lazy().filter(pl.col("msg") == "200").select(pl.col("user_id")
,pl.col("num_xact_of_users"),pl.col("num_share_of_users"))
.collect())
msg_200 = msg_200.to_pandas()
pg.normality(msg_200["num_xact_of_users"])


,W,pval,normal
num_xact_of_users,0.800873,2.349948e-43,False


In [25]:
msg_200


ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
315,0,"""200""",0,0.91,1.82
315,0,"""200""",1,1.0,1.7
315,0,"""200""",2,1.0,1.52
315,0,"""200""",3,1.0,1.98
315,0,"""200""",4,1.0,2.7
…,…,…,…,…,…
163,1,"""200""",470,0.71,0.0
163,1,"""200""",471,0.0,0.0
163,1,"""200""",472,0.64,0.0
